# Decision models as rerankers, on a free Colab GPU

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/treygrainger/ai-powered-search/blob/main/course/decision-models-as-rerankers/colab_free_tier.ipynb)

The companion notebook to the talk *Decision Models as Rerankers* (Adam Hevenor, [hevmind.com](https://hevmind.com)), run on an open-weight decision model on the **free Colab T4, with no API keys**.
It follows the deck: **Part A** builds the three request shapes from Part 2 of the deck by hand, on the slides'
six-passage example, and sends them straight to the model's own reference code. **Part B** does the same with
[`hev-rerank`](https://github.com/hev/reranker) in one line per shape, then benchmarks the shapes on SciFact (Part 3) and
writes a custom definition of relevant (Part 4). Headings name the deck's parts, so you can find your place from a
slide.

**Model:** [`perplexity-ai/pplx-decider-flash-v1.3`](https://huggingface.co/perplexity-ai/pplx-decider-flash-v1.3),
a 4B decision model distilled from Perplexity's 27B one (Apache-2.0). The slides use Jev, TypeSafe's hosted model;
this model answers the same typed questions, so its numbers are close but not identical.

**What you'll need:** a Colab runtime with a T4: *Runtime → Change runtime type → T4 GPU*. Then *Runtime → Run all*.
No keys, no account. **How long:** 11 minutes in our cold-start run on a T4 (an AWS g4dn.xlarge, the free-Colab card):
2 installing, 2 downloading and loading the 8.5 GB of weights, 6 for Part 3. Allow up to 15 when the install is
slow.

With keys, [hosted_walkthrough.ipynb](https://colab.research.google.com/github/treygrainger/ai-powered-search/blob/main/course/decision-models-as-rerankers/hosted_walkthrough.ipynb)
runs the slides' exact code on Jev and compares five hosted models.

In [1]:
# Parameters. HEV_RERANK_REF is the git tag or commit of hev-rerank to install.
HEV_RERANK_REF = "v0.3.0"
MODEL = "perplexity-ai/pplx-decider-flash-v1.3"
REVISION = "d06fb3f2d97e0a52325a0d463c450cf0338f44fe"   # the pinned weights this notebook ran with
N_QUERIES = 12      # SciFact test queries per shape in Part 3: sized so the notebook finishes in ~15 minutes on a T4
K = 20              # BM25 candidates per query to rerank
BATCH_DOCS = 5      # documents per batch request on the T4 (see the note under the Part 3 results)

In [2]:
# Parameters
HEV_RERANK_REF = "v0.3.0"


In [3]:
# Installs torch, transformers and hev-rerank. hev-rerank isn't imported until Part B.
%pip install -q "hev-rerank[local] @ git+https://github.com/hev/reranker@{HEV_RERANK_REF}" rank_bm25
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"   # fewer out-of-memory errors from fragmentation

Note: you may need to restart the kernel to use updated packages.


In [4]:
import torch, transformers
assert torch.cuda.is_available(), "No GPU: Runtime -> Change runtime type -> T4 GPU, then run all again."
gpu = torch.cuda.get_device_name(0)
print(f"torch {torch.__version__}, transformers {transformers.__version__}")
print(f"GPU: {gpu}, {torch.cuda.get_device_properties(0).total_memory / 2**30:.1f} GiB, "
      f"bf16: {torch.cuda.is_bf16_supported(including_emulation=False)}")

/home/ubuntu/nbvenv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


torch 2.15.0+cu132, transformers 5.23.0
GPU: Tesla T4, 14.6 GiB, bf16: False


## Part 1 · History: retrieve, then rerank

BM25 (or any first-stage retriever) is cheap enough to score a whole corpus, so it *retrieves*: it cuts millions of
documents down to a shortlist of 20–100. A reranker is too expensive for the whole corpus, so it only *reorders the
cut*. Two metrics tell those jobs apart:

- **Recall@k**: how many of the relevant documents made it into the top k. For the shortlist as a whole this is fixed
  by the retriever; a reranker can't bring back a document BM25 missed.
- **nDCG@k**: whether the relevant documents are *at the top*, discounted by rank. Reordering the cut is exactly what
  moves it (and recall@5 too, by pulling relevant documents up from places 6–30).

Part 3 measures them on SciFact: BM25's recall over the shortlist (the ceiling), and nDCG@10 before and after
reranking.

# Part A · By hand

The three request shapes from Part 2 of the deck, written out by hand on the deck's six-passage example. No
`hev-rerank` yet: it's imported in Part B.

## Part 2 · The fit: load a decision model

A decision model takes **state** (any text or JSON) plus **typed questions** about it, and returns calibrated
probabilities instead of generated text. The checkpoint ships its own inference code, `pplx_decider`; its
`decide()` takes the same request the hosted APIs do: `{"state": ..., "questions": {...}}`.

This cell downloads the weights (8.5 GB, once) and loads them on the GPU. Part B reuses this loaded model.

**Two T4 details.** The reference loader always uses bfloat16 on a GPU, which the T4 doesn't have, so we load the
backbone in float16 instead. And on the first request `transformers` warns that `causal_conv1d` and
`flash-linear-attention` aren't installed and falls back to plain PyTorch: that's expected, on a T4
`flash-linear-attention` measured slower than the fallback.

In [5]:
import json, sys, time
from pathlib import Path
from huggingface_hub import snapshot_download

t0 = time.perf_counter()
checkpoint = Path(snapshot_download(MODEL, revision=REVISION,       # decision weights + the reference code
                                    allow_patterns=["*.json", "*.jinja", "*.safetensors", "*.py", "LICENSE", "NOTICE", "source/**"],
                                    ignore_patterns=["locate-span/*"]))  # skip the span-extraction adapter
sys.path.insert(0, str(checkpoint / "source" / "src"))
import pplx_decider.model
from pplx_decider import Decider

# T4: no bfloat16. Load the backbone in float16 (bf16 where the GPU has it) and cast the readout head to match.
dtype = torch.bfloat16 if torch.cuda.is_bf16_supported(including_emulation=False) else torch.float16
# Keep the library's original loader in one place, so re-running this cell can't wrap the wrapper.
_backbone = getattr(pplx_decider.model, "_original_backbone", pplx_decider.model._backbone)
pplx_decider.model._original_backbone = _backbone
pplx_decider.model._backbone = lambda path, config, _bf16: _backbone(path, config, dtype)
try:
    decider = Decider.from_pretrained(checkpoint, device="cuda", mode="isolated", locate=False)
finally:
    pplx_decider.model._backbone = _backbone
decider.readout.to(dtype)

load_seconds = time.perf_counter() - t0
print(f"{MODEL} in {dtype}: download + load {load_seconds:.0f} s, GPU memory {torch.cuda.memory_allocated() / 2**30:.1f} GiB")

Fetching 22 files:   0%|          | 0/22 [00:00<?, ?it/s]

Fetching 22 files:  41%|████      | 9/22 [00:00<00:00, 85.18it/s]

Fetching 22 files:  86%|████████▋ | 19/22 [00:00<00:00, 66.33it/s]

Fetching 22 files:  91%|█████████ | 20/22 [00:19<00:00, 66.33it/s]

Fetching 22 files:  95%|█████████▌| 21/22 [01:07<00:05,  5.04s/it]

Fetching 22 files: 100%|██████████| 22/22 [01:19<00:00,  5.69s/it]

... (32 outputs left out)

Loading weights:  98%|█████████▊| 708/723 [00:26<00:00, 45.07it/s]

Loading weights:  99%|█████████▉| 716/723 [00:26<00:00, 39.78it/s]

Loading weights: 100%|██████████| 723/723 [00:26<00:00, 27.01it/s]

perplexity-ai/pplx-decider-flash-v1.3 in torch.float16: download + load 122 s, GPU memory 8.5 GiB


`mode="isolated"` reads a request's state once and answers every question from it (the layout the model was trained
on); it's what makes one batch call cheaper than six pairwise calls.

The deck's example: a query and six search results, keyed `D00`…`D05`. Two passages answer the question, one
for (D04, a trial) and one against (D00, a meta-analysis); relevance is *addresses the query*, not *agrees with it*.
D03 is related background.

The slides use TypeSafe's SDK, where `Noul(...)` and `Choice(...)` build typed questions. The decider's `decide()`
takes the same questions as plain dicts, so here `Noul` and `Choice` are two small functions that build them: the
code below reads like the slides.

In [6]:
query = "does vitamin D supplementation improve bone density in older adults"
docs = {"D00": "Meta-analysis of 23 trials: vitamin D alone did not meaningfully increase bone density in older adults.",
        "D01": "Dietary sources of vitamin D include oily fish, egg yolk and fortified milk. Many older adults fall short.",
        "D02": "Sunscreen reduces vitamin D synthesis in the skin only marginally in practice.",
        "D03": "Vitamin D increases intestinal calcium absorption. Deficiency impairs bone mineralization.",
        "D04": "Randomised trial: daily vitamin D3 with calcium raised hip bone density in adults over 65 versus placebo.",
        "D05": "Astronauts lose 1 to 2 percent of bone density per month in microgravity."}   # query results

part_a = {}   # each shape's scores, kept to compare with hev-rerank in Part B

def show(scores, ranked):
    for d in ranked:
        print(f"  {d}  {scores[d]:5.2f}  {docs[d][:85]}")

def Noul(instructions, criteria=None):        # a true-or-false question, answered as a probability
    return {"type": "noul", "instructions": instructions, "criteria": criteria}

def Choice(instructions, criteria):           # one question, one option per key of criteria
    return {"type": "choice", "instructions": instructions, "criteria": criteria}

def show_request(request):
    """The request as the model gets it: the state, then each typed question."""
    print("state =", json.dumps(request["state"], indent=2))
    print("questions =")
    for key, q in request["questions"].items():
        print(f"  {key}: {q['type']}, {q['instructions']!r}")
        for option, text in (q["criteria"] or {}).items():
            print(f"      {option}: {text}")

## Part 2 · Pairwise: one call per document

Slide *"Typed questions in, probabilities out"*. One request per (query, document) with one **Noul**, a true-or-false
question answered as a probability. The answer is an absolute `p_relevant`: it means the same thing whatever else is
in the list.

In [7]:
def pair_request(doc: str) -> dict:
    return {"state": {"query": query, "document": doc},
            "questions": {"relevant": Noul(
                instructions="`document` is relevant to `query`: "
                             "it contains information that answers or directly addresses it.",
                criteria={"true":  "The document contains information that answers the query "
                                   "or directly addresses what it asks about.",
                          "false": "The document is only loosely related, on a similar topic, "
                                   "or does not address what the query asks."})}}

show_request(pair_request(docs["D00"]))       # the request for D00; the other five differ only in `document`

state = {
  "query": "does vitamin D supplementation improve bone density in older adults",
  "document": "Meta-analysis of 23 trials: vitamin D alone did not meaningfully increase bone density in older adults."
}
questions =
  relevant: noul, '`document` is relevant to `query`: it contains information that answers or directly addresses it.'
      true: The document contains information that answers the query or directly addresses what it asks about.
      false: The document is only loosely related, on a similar topic, or does not address what the query asks.


In [8]:
def relevance(doc: str) -> float:             # one call per document
    r = decider.decide(pair_request(doc))
    return r["answers"]["relevant"]["noul"]

t = time.perf_counter()
p_relevant = {d: relevance(text) for d, text in docs.items()}
ranked = sorted(docs, key=p_relevant.get, reverse=True)
print(f"pairwise, 6 calls, {time.perf_counter() - t:.1f} s:")
show(p_relevant, ranked)
part_a["pair"] = p_relevant

[transformers] `causal_conv1d_fn` is falling back to its reference PyTorch implementation because `causal_conv1d` is not installed. This is correct but much slower; install `causal_conv1d` for the optimized kernel.


[transformers] `chunk_gated_delta_rule` is falling back to its reference PyTorch implementation because `flash-linear-attention` is not installed. This is correct but much slower; install `flash-linear-attention` for the optimized kernel.


pairwise, 6 calls, 5.0 s:
  D00   1.00  Meta-analysis of 23 trials: vitamin D alone did not meaningfully increase bone densit
  D04   0.99  Randomised trial: daily vitamin D3 with calcium raised hip bone density in adults ove
  D03   0.61  Vitamin D increases intestinal calcium absorption. Deficiency impairs bone mineraliza
  D01   0.01  Dietary sources of vitamin D include oily fish, egg yolk and fortified milk. Many old
  D02   0.00  Sunscreen reduces vitamin D synthesis in the skin only marginally in practice.
  D05   0.00  Astronauts lose 1 to 2 percent of bone density per month in microgravity.


## Part 2 · Listwise batch: one call, every document

Slide *"One call, every document"*. All six passages go into one state, keyed `D00`…`D05`, with one Noul per
document that points at its key (`documents.D00`). Still an absolute `p_relevant` per document, from one call.

In [9]:
questions = {
    d: Noul(  # a true-or-false question, answered as a probability
        instructions=f"Document `documents.{d}` is relevant to `query`: "
                     "it contains information that answers or directly addresses it.",
        criteria={"true":  "The document contains information that answers the query "
                           "or directly addresses what it asks about.",
                  "false": "The document is only loosely related, on a similar topic, "
                           "or does not address what the query asks."})
    for d in docs}

request = {"state": {"query": query, "documents": docs}, "questions": questions}
show_request(request)

state = {
  "query": "does vitamin D supplementation improve bone density in older adults",
  "documents": {
    "D00": "Meta-analysis of 23 trials: vitamin D alone did not meaningfully increase bone density in older adults.",
    "D01": "Dietary sources of vitamin D include oily fish, egg yolk and fortified milk. Many older adults fall short.",
    "D02": "Sunscreen reduces vitamin D synthesis in the skin only marginally in practice.",
    "D03": "Vitamin D increases intestinal calcium absorption. Deficiency impairs bone mineralization.",
    "D04": "Randomised trial: daily vitamin D3 with calcium raised hip bone density in adults over 65 versus placebo.",
    "D05": "Astronauts lose 1 to 2 percent of bone density per month in microgravity."
  }
}
questions =
  D00: noul, 'Document `documents.D00` is relevant to `query`: it contains information that answers or directly addresses it.'
      true: The document contains information that answers the query or directly addresses what it ask

In [10]:
t = time.perf_counter()
r = decider.decide(request)                   # one call

p_relevant = {d: r["answers"][d]["noul"] for d in docs}
ranked = sorted(docs, key=p_relevant.get, reverse=True)
print(f"listwise batch, 1 call, {time.perf_counter() - t:.1f} s:")
show(p_relevant, ranked)
part_a["batch"] = p_relevant

listwise batch, 1 call, 1.1 s:
  D00   0.99  Meta-analysis of 23 trials: vitamin D alone did not meaningfully increase bone densit
  D04   0.99  Randomised trial: daily vitamin D3 with calcium raised hip bone density in adults ove
  D03   0.47  Vitamin D increases intestinal calcium absorption. Deficiency impairs bone mineraliza
  D01   0.02  Dietary sources of vitamin D include oily fish, egg yolk and fortified milk. Many old
  D05   0.00  Astronauts lose 1 to 2 percent of bone density per month in microgravity.
  D02   0.00  Sunscreen reduces vitamin D synthesis in the skin only marginally in practice.


## Part 2 · Listwise choice: which is best?

Slide *"One call, one question: which is best?"*. One **Choice** whose options are the documents. The answer is
`p_best`: one unit of probability spread across the list, so it sums to one.

In [11]:
request = {"state": {"query": query, "documents": docs},
           "questions": {"best": Choice(  # one question, one option per document
               instructions="Which document in `documents` best answers or most directly addresses `query`?",
               criteria={d: f"documents.{d}" for d in docs})}}
show_request(request)

state = {
  "query": "does vitamin D supplementation improve bone density in older adults",
  "documents": {
    "D00": "Meta-analysis of 23 trials: vitamin D alone did not meaningfully increase bone density in older adults.",
    "D01": "Dietary sources of vitamin D include oily fish, egg yolk and fortified milk. Many older adults fall short.",
    "D02": "Sunscreen reduces vitamin D synthesis in the skin only marginally in practice.",
    "D03": "Vitamin D increases intestinal calcium absorption. Deficiency impairs bone mineralization.",
    "D04": "Randomised trial: daily vitamin D3 with calcium raised hip bone density in adults over 65 versus placebo.",
    "D05": "Astronauts lose 1 to 2 percent of bone density per month in microgravity."
  }
}
questions =
  best: choice, 'Which document in `documents` best answers or most directly addresses `query`?'
      D00: documents.D00
      D01: documents.D01
      D02: documents.D02
      D03: documents.D03
      D04: documents.D04
      D

In [12]:
t = time.perf_counter()
r = decider.decide(request)                   # one call

p_best = r["answers"]["best"]["probabilities"]
ranked = sorted(docs, key=p_best.get, reverse=True)
print(f"listwise choice, 1 call, {time.perf_counter() - t:.1f} s (p_best sums to {sum(p_best.values()):.3f}):")
show(p_best, ranked)
part_a["choice"] = p_best

listwise choice, 1 call, 0.2 s (p_best sums to 1.000):
  D00   0.82  Meta-analysis of 23 trials: vitamin D alone did not meaningfully increase bone densit
  D04   0.17  Randomised trial: daily vitamin D3 with calcium raised hip bone density in adults ove
  D03   0.01  Vitamin D increases intestinal calcium absorption. Deficiency impairs bone mineraliza
  D01   0.00  Dietary sources of vitamin D include oily fish, egg yolk and fortified milk. Many old
  D05   0.00  Astronauts lose 1 to 2 percent of bone density per month in microgravity.
  D02   0.00  Sunscreen reduces vitamin D synthesis in the skin only marginally in practice.


## Part 2 · When to use which

| shape | calls for N docs | score | threshold? | use it when |
|---|---|---|---|---|
| **pairwise** | N | `p_relevant`, absolute | yes | you want the most robust per-document judgement and calls are cheap or parallel |
| **listwise batch** | 1 (or a few) | `p_relevant`, absolute | yes | you want pairwise's absolute scores at a fraction of the calls; the model also sees the other candidates |
| **listwise choice** | 1, always | `p_best`, sums to one | **no** | you need the single best answer, or few documents are relevant |

Pairwise and batch say *how likely is this one relevant*, so several documents can be near 1 (D00 and D04 here) and a
threshold prunes the rest. Choice says *which one wins*: D00 and D04 split one unit between them, and a document's
`p_best` depends on what else is in the list, so a fixed cut-off means nothing. Which shape ranks best depends on the
model: benchmark the shape on yours (Part 3).

# Part B · The same three shapes with hev-rerank

[`hev-rerank`](https://github.com/hev/reranker) wraps everything in Part A behind one call:
`rerank(query, docs, model=..., style=...)`, returning `RerankResult(index, score, document)` best-first. `style=` is
the shape: `"pair"`, `"batch"` or `"choice"`.

`prompt=` sets the wording. `slides` below is Part A's wording, so Part B asks exactly Part A's questions. (The
library's own default wording differs slightly per model; Part 3 uses the default.)

In [13]:
import hev_rerank
from hev_rerank import local, rerank
print(f"hev-rerank {hev_rerank.__version__}")

# rerank(model="local:...") loads the weights itself on first use; client= hands it the model Part A already loaded.
model = local.LocalModel(MODEL, decider, "cuda", dtype, "isolated")

slides = {
    "pair_question": "`document` is relevant to `query`: it contains information that answers or directly addresses it.",
    "question": "Document `documents.{id}` is relevant to `query`: it contains information that answers or directly addresses it.",
    "choice_question": "Which document in `documents` best answers or most directly addresses `query`?",
    "criteria": {"true": "The document contains information that answers the query or directly addresses what it asks about.",
                 "false": "The document is only loosely related, on a similar topic, or does not address what the query asks."},
}
ids, texts = list(docs), list(docs.values())    # rerank takes a list; its result's .index is the position in it

def compare(style, hits):
    """Part B's scores and order next to Part A's for the same shape."""
    b = {ids[h.index]: h.score for h in hits}
    a = part_a[style]
    order_a, order_b = sorted(ids, key=a.get, reverse=True), [ids[h.index] for h in hits]
    print(f"{style}:  doc   Part A  Part B")
    for d in order_b:
        print(f"         {d}  {a[d]:6.3f}  {b[d]:6.3f}")
    print(f"  Part A order {order_a}\n  Part B order {order_b}   same order: {order_a == order_b}, "
          f"max |score difference| {max(abs(a[d] - b[d]) for d in ids):.3f}\n")

hev-rerank 0.3.0


In [14]:
for style in ["pair", "batch", "choice"]:
    hits = rerank(query, texts, model=f"local:{MODEL}", style=style, prompt=slides, client=model)
    compare(style, hits)

pair:  doc   Part A  Part B
         D00   0.997   0.997
         D04   0.990   0.990
         D03   0.610   0.611
         D01   0.013   0.013
         D02   0.002   0.002
         D05   0.002   0.002
  Part A order ['D00', 'D04', 'D03', 'D01', 'D02', 'D05']
  Part B order ['D00', 'D04', 'D03', 'D01', 'D02', 'D05']   same order: True, max |score difference| 0.001



batch:  doc   Part A  Part B
         D00   0.994   0.994
         D04   0.993   0.993
         D03   0.471   0.471
         D01   0.023   0.023
         D05   0.005   0.005
         D02   0.003   0.003
  Part A order ['D00', 'D04', 'D03', 'D01', 'D05', 'D02']
  Part B order ['D00', 'D04', 'D03', 'D01', 'D05', 'D02']   same order: True, max |score difference| 0.000



choice:  doc   Part A  Part B
         D00   0.821   0.821
         D04   0.166   0.166
         D03   0.006   0.006
         D01   0.003   0.003
         D05   0.002   0.002
         D02   0.002   0.002
  Part A order ['D00', 'D04', 'D03', 'D01', 'D05', 'D02']
  Part B order ['D00', 'D04', 'D03', 'D01', 'D05', 'D02']   same order: True, max |score difference| 0.000



Each `rerank(...)` call is one shape from Part A: same model, same questions, same order. The scores can differ
in the third decimal: `hev-rerank` sends the six pairwise requests to the GPU as one padded batch rather than six
calls, and float16 arithmetic on padded batches isn't bit-identical.

## Part 3 · Benchmark: SciFact, BM25 then rerank

SciFact (from BEIR) has 5,183 paper abstracts and 300 test claims with human relevance labels. BM25 retrieves the top
K for each claim; each shape reranks them; nDCG@10 is measured before and after. From here on `rerank` uses the
library's default wording for this model (the one it was benchmarked with).

In [15]:
import csv, io, json, math, random, re, urllib.request, zipfile
from rank_bm25 import BM25Okapi

# SciFact from BEIR (5,183 abstracts, 300 test queries with human relevance labels), 2.8 MB.
raw = urllib.request.urlopen("https://public.ukp.informatik.tu-darmstadt.de/thakur/BEIR/datasets/scifact.zip").read()
z = zipfile.ZipFile(io.BytesIO(raw))
corpus = {d["_id"]: {"title": d["title"], "text": d["text"]} for d in map(json.loads, z.open("scifact/corpus.jsonl"))}
queries = {q["_id"]: q["text"] for q in map(json.loads, z.open("scifact/queries.jsonl"))}
qrels = {}
for row in csv.DictReader(io.TextIOWrapper(z.open("scifact/qrels/test.tsv")), delimiter="\t"):
    qrels.setdefault(row["query-id"], {})[row["corpus-id"]] = int(row["score"])

tok = lambda s: re.findall(r"\w+", s.lower())
corpus_ids = list(corpus)
bm25 = BM25Okapi([tok(corpus[i]["title"] + " " + corpus[i]["text"]) for i in corpus_ids])

def bm25_top(q, k=K):
    scores = bm25.get_scores(tok(q))
    return [corpus_ids[i] for i in sorted(range(len(corpus_ids)), key=lambda i: -scores[i])[:k]]

def ndcg10(ranked, rel):
    dcg = sum(rel.get(d, 0) / math.log2(i + 2) for i, d in enumerate(ranked[:10]))
    ideal = sum(r / math.log2(i + 2) for i, r in enumerate(sorted(rel.values(), reverse=True)[:10]))
    return dcg / ideal if ideal else 0.0

qids = sorted(qrels)[:N_QUERIES]          # the first N test queries, sorted by id: the same subset every run
shortlists = {q: bm25_top(queries[q]) for q in qids}
bm25_ndcg = sum(ndcg10(shortlists[q], qrels[q]) for q in qids) / len(qids)
recall = sum(len(set(shortlists[q]) & set(qrels[q])) / len(qrels[q]) for q in qids) / len(qids)
print(f"{len(qids)} SciFact test queries, BM25 top-{K}: nDCG@10 {bm25_ndcg:.3f}, recall@{K} {recall:.3f} "
      "(a reranker can only reorder what BM25 found)")

12 SciFact test queries, BM25 top-20: nDCG@10 0.653, recall@20 0.806 (a reranker can only reorder what BM25 found)


In [16]:
def pct(xs, p):
    xs = sorted(xs)
    return xs[min(len(xs) - 1, int(round(p / 100 * (len(xs) - 1))))]

results = {}
for style in ["pair", "batch", "choice"]:
    latencies, ndcgs, t0 = [], [], time.perf_counter()
    for q in qids:
        cands = [corpus[d] for d in shortlists[q]]
        torch.cuda.synchronize(); t = time.perf_counter()
        hits = rerank(queries[q], cands, model=f"local:{MODEL}", style=style, client=model,
                      max_docs_per_call=BATCH_DOCS if style == "batch" else K)
        torch.cuda.synchronize(); latencies.append(time.perf_counter() - t)
        ndcgs.append(ndcg10([shortlists[q][h.index] for h in hits], qrels[q]))
    results[style] = {"ndcg": sum(ndcgs) / len(ndcgs), "p50": pct(latencies, 50), "p95": pct(latencies, 95),
                      "wall": time.perf_counter() - t0}
    r = results[style]
    print(f"{style:6s}  nDCG@10 {bm25_ndcg:.3f} -> {r['ndcg']:.3f}   per query p50 {r['p50']:.2f} s, "
          f"p95 {r['p95']:.2f} s   wall {r['wall']:.0f} s")

pair    nDCG@10 0.653 -> 0.767   per query p50 9.10 s, p95 10.51 s   wall 108 s


batch   nDCG@10 0.653 -> 0.769   per query p50 16.21 s, p95 18.87 s   wall 193 s


choice  nDCG@10 0.653 -> 0.804   per query p50 5.04 s, p95 6.09 s   wall 60 s


In [17]:
from IPython.display import Markdown
rows = [f"| {s} | {bm25_ndcg:.3f} | {r['ndcg']:.3f} | {r['ndcg'] - bm25_ndcg:+.3f} | {r['p50']:.2f} | {r['p95']:.2f} | {r['wall']:.0f} |"
        for s, r in results.items()]
Markdown(f"**{MODEL}** on {gpu}, {len(qids)} SciFact test queries, BM25 top-{K}, {str(dtype).split('.')[-1]}\n\n"
         "| shape | nDCG@10 BM25 | nDCG@10 reranked | Δ | p50 s/query | p95 s/query | wall s |\n|---|---|---|---|---|---|---|\n"
         + "\n".join(rows))

**perplexity-ai/pplx-decider-flash-v1.3** on Tesla T4, 12 SciFact test queries, BM25 top-20, float16

| shape | nDCG@10 BM25 | nDCG@10 reranked | Δ | p50 s/query | p95 s/query | wall s |
|---|---|---|---|---|---|---|
| pair | 0.653 | 0.767 | +0.114 | 9.10 | 10.51 | 108 |
| batch | 0.653 | 0.769 | +0.116 | 16.21 | 18.87 | 193 |
| choice | 0.653 | 0.804 | +0.150 | 5.04 | 6.09 | 60 |

**About these latencies.** They are per query (all K candidates) on a free T4, with the checkpoint's reference
PyTorch code: the T4 has no bfloat16 and no FlashAttention, and the decider's optimized path (packed sequences,
CUDA graphs) needs an Ampere or newer GPU. They are **not comparable with hosted API latencies**, which run the same
kind of model on datacenter GPUs with batching across users, and include the network. For hosted numbers see
the [benchmark](https://hevmind.com/research/rerankers). Use this notebook to compare the *shapes* with one
another on one machine, and to see the quality.

**Why batch is split into requests of five here.** This checkpoint's reference code answers each extra question
in a request by re-running its linear-attention layers over the whole shared state, so on this path a batch
request's cost grows with the square of its length, and twenty documents in one request run out of the T4's
memory. Five per request is about as fast as it gets on a T4. Batch scores are absolute probabilities, so scores
from separate requests stay comparable. (Hosted, and on Ampere GPUs with the checkpoint's fast path, the state is
read once and batch costs about the same as choice.) Choice stays one request over all candidates: its
probabilities are only comparable within one list.

With a dozen queries the nDCG@10 differences between shapes are noise-sized; the
[rerank-bench](https://github.com/hev/rerank-bench) numbers use all 300 SciFact queries plus NFCorpus and FiQA.

## Choice can't be thresholded

Pair and batch scores are calibrated probabilities, so a threshold prunes candidates the model is confident are
irrelevant. Choice scores sum to one across the list, so `hev-rerank` refuses a threshold on them.

In [18]:
q = qids[0]
cands = [corpus[d] for d in shortlists[q]]
kept = rerank(queries[q], cands, model=f"local:{MODEL}", style="batch", client=model, max_docs_per_call=BATCH_DOCS, threshold=0.1)
print(f"{queries[q]!r}: batch keeps {len(kept)} of {len(cands)} candidates at threshold 0.1")
try:
    rerank(queries[q], cands, model=f"local:{MODEL}", style="choice", client=model, threshold=0.1)
except ValueError as e:
    print("ValueError:", e)

'0-dimensional biomaterials show inductive properties.': batch keeps 2 of 20 candidates at threshold 0.1
ValueError: threshold can't be used with style='choice': choice probabilities sum to one across the list, so they rank documents but can't be thresholded. Use style='pair' or 'batch'.


## Part 4 · Customize: your own definition of relevant

The question and its true/false criteria are plain text: change them to say what *relevant* means for your corpus.
`prompt=` overrides any of `question` (batch, with `{id}` for the document id), `pair_question`, `choice_question` and
`criteria`; anything you leave out keeps the default. Here, a stricter notion: only **clinical trial evidence**
counts, so the mechanism passage (D03) drops while the trial (D04) and the meta-analysis (D00) stay.

The deck also covers **intent**: what the person searching is trying to get done, which can change what counts as
relevant for the same query (a clinician choosing a treatment, a student writing a review). It's a concept, not a
`hev-rerank` parameter; the place to express it is the same as here, the wording of the question and criteria.

In [19]:
strict = {
    "question": "Does document `documents.{id}` report trial or meta-analysis evidence on the question in `query`?",
    "pair_question": "Does the document report trial or meta-analysis evidence on the question in the query?",
    "criteria": {"true": "The document reports results of a trial or a meta-analysis on the query's question, whatever the result.",
                 "false": "Background, mechanisms, dietary advice or other populations: not trial evidence on the question."},
}

default = {ids[h.index]: h.score for h in rerank(query, texts, model=f"local:{MODEL}", style="batch", prompt=slides, client=model)}
custom = {ids[h.index]: h.score for h in rerank(query, texts, model=f"local:{MODEL}", style="batch", prompt=strict, client=model)}
print("batch, slides' wording -> trial-evidence wording")
for d in sorted(custom, key=custom.get, reverse=True):
    print(f"  {d}  {default[d]:6.3f} -> {custom[d]:6.3f}  {docs[d][:70]}")

batch, slides' wording -> trial-evidence wording
  D00   0.994 ->  0.998  Meta-analysis of 23 trials: vitamin D alone did not meaningfully incre
  D04   0.993 ->  0.998  Randomised trial: daily vitamin D3 with calcium raised hip bone densit
  D03   0.471 ->  0.002  Vitamin D increases intestinal calcium absorption. Deficiency impairs 
  D05   0.005 ->  0.001  Astronauts lose 1 to 2 percent of bone density per month in microgravi
  D02   0.003 ->  0.001  Sunscreen reduces vitamin D synthesis in the skin only marginally in p
  D01   0.023 ->  0.001  Dietary sources of vitamin D include oily fish, egg yolk and fortified


## Part 5 · Conclusions

- A decision model is a reranker once you ask it *"is this document relevant to the query?"* as a typed question:
  state in, calibrated probabilities out.
- Three request shapes: **pairwise** (one call per document), **listwise batch** (one call, one Noul per document) and
  **listwise choice** (one call, one Choice). Pairwise and batch give absolute probabilities you can threshold; choice
  gives relative ones you can only sort.
- The best shape depends on the model, so benchmark it on yours: with `hev-rerank` that's one argument.
- *Relevant* is a prompt. Write your own definition.

More: the benchmark at [hevmind.com/research/rerankers](https://hevmind.com/research/rerankers), code at
[hev/rerank-bench](https://github.com/hev/rerank-bench), and this library at
[hev/reranker](https://github.com/hev/reranker).

## Optional: bigger models (not free tier)

The cell below is **off by default and will not fit a free T4**. It shows how to run larger open-weight decision
models with the same call, on a paid Colab GPU:

- [`Cloudflare/clef-flash`](https://huggingface.co/Cloudflare/clef-flash) (9B, Apache-2.0) in bfloat16 needs about
  20 GB: an **L4** (24 GB) or A100. Its best measured shape is pair.
- 27B models ([`Cloudflare/clef`](https://huggingface.co/Cloudflare/clef), Apache-2.0) in **4-bit** need about
  18 GB with `bitsandbytes`: an L4 or A100.
- `perplexity-ai/pplx-decider-v1.3` (27B) runs in bfloat16 on an 80 GB A100/H100.

This cell was not executed for this notebook's saved outputs.

In [20]:
RUN_BIGGER_MODELS = False   # set True on an L4 / A100 runtime (Colab Pro); not free tier

if RUN_BIGGER_MODELS:
    del model, decider; local._LOADED.clear(); torch.cuda.empty_cache()

    # Clef-flash 9B, bfloat16 on an L4 or A100.
    big = local.load("Cloudflare/clef-flash")
    for h in rerank(query, texts, model="local:Cloudflare/clef-flash", style="pair", client=big):
        print(f"  {h.score:6.3f}  {ids[h.index]}  {texts[h.index][:90]}")
    del big; torch.cuda.empty_cache()

    # A 27B decision model in 4-bit (pip install bitsandbytes).
    from transformers import BitsAndBytesConfig
    q4 = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=torch.bfloat16, bnb_4bit_quant_type="nf4")
    big = local.load("Cloudflare/clef", quantization_config=q4)
    for h in rerank(query, texts, model="local:Cloudflare/clef", style="pair", client=big):
        print(f"  {h.score:6.3f}  {ids[h.index]}  {texts[h.index][:90]}")